# 03 · Análisis SQL y anotación de Schema Linking

Objetivos:

1. Construir el AST 'Abstract Syntax Tree (árbol de sintaxis abstracta)' de cada SQL gold con **SQLGlot** (dialecto `sqlite` para Spider y BIRD, `postgres` para Catastro).
2. Resolver semánticamente tablas y columnas contra el esquema real (alias, columnas calificadas y no calificadas, subconsultas, CTE, UNION).
3. Generar, para cada pregunta y **cada elemento de su esquema**, la etiqueta de relevancia y el vector de 7 roles
   `[SELECT, WHERE, JOIN, FROM, GROUP_BY, HAVING, ORDER_BY]`.

Toda la lógica está en `src/sql_parser.py` y `src/schema_annotator.py`.


In [2]:
# Configuración común: localizar la raíz del proyecto y habilitar `src/`
import sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import json
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

from src import config
config.ensure_directories()
print("Raíz del proyecto:", ROOT.name)

Raíz del proyecto: proyecto_jarm


In [3]:
from src.data_loader import load_master_dataset
from src.schema_parser import load_schema_catalog
from src.sql_parser import SchemaIndex, analyze_sql, analyze_dataset, parse_sql

master = load_master_dataset()
catalog = load_schema_catalog()
print(len(master), "preguntas |", len(catalog), "esquemas")

19076 preguntas | 247 esquemas


## 1. Análisis AST con SQLGlot

Ejemplo de árbol sintáctico de una consulta Spider:

In [4]:
ex = master[master.example_id == "SPIDER-dev-00010"].iloc[0]
print(ex.question); print(ex.sql_gold)
tree, err = parse_sql(ex.sql_gold, ex.dialect)
print("\nSQL normalizado:", tree.sql(dialect="sqlite", pretty=True))
print("\nAST (repr abreviado):"); print(repr(tree)[:1200])

Show all countries and the number of singers in each country.
SELECT country ,  count(*) FROM singer GROUP BY country

SQL normalizado: SELECT
  country,
  COUNT(*)
FROM singer
GROUP BY
  country

AST (repr abreviado):
Select(
  expressions=[
    Column(
      this=Identifier(this=country, quoted=False)),
    Count(
      this=Star(),
      big_int=True)],
  from_=From(
    this=Table(
      this=Identifier(this=singer, quoted=False))),
  group=Group(
    expressions=[
      Column(
        this=Identifier(this=country, quoted=False))]))


## 2. Metodología de resolución semántica

Para cada consulta se recorren sus **bloques** (consulta principal, subconsultas, CTE y ramas de operaciones de conjunto) y:

| Situación | Regla aplicada |
|---|---|
| Tabla en `FROM` | rol `FROM`; si el bloque tiene `JOIN` (explícito o implícito con coma) recibe también `JOIN` |
| Tabla incorporada con `JOIN` | roles `FROM` + `JOIN` |
| Columna calificada (`T1.col`, `tabla.col`, `schema.tabla.col`) | se resuelve el alias en el bloque actual y, si no existe, en los bloques externos (subconsultas correlacionadas) |
| Columna no calificada | candidatas = tablas físicas **seleccionadas** en el bloque + salidas de CTE/subconsultas; si hay exactamente una → resuelta; si hay varias → `unresolved: ambiguous` (nunca se escoge la primera) |
| Alias de proyección (`ORDER BY cnt`) | se rastrea al origen físico de la expresión (linaje) |
| Columna de un CTE o subconsulta (`tb1.numero_predial`) | se propaga el rol de la cláusula externa a la(s) columna(s) física(s) de origen (`via_derived`) |
| `*` / `COUNT(*)` | no genera columnas (índice -1); se registra la tabla implicada |
| `"texto"` en SQLite sin columna homónima | literal de texto (semántica real de SQLite): `literal_fallback`, no es error |
| Tabla/columna inexistente en el catálogo | `unresolved` con motivo (`table_not_in_schema`, `column_not_in_table`, ...) |

La cláusula se determina subiendo por los ancestros del nodo hasta el `SELECT` que lo contiene: los roles son **locales a cada bloque**
(una columna dentro de `WHERE x = (SELECT MAX(y) ...)` recibe `SELECT` para `y` y `WHERE` para `x`).


## 3. Ejemplos ilustrativos

In [5]:
def show_analysis(example_id):
    ex = master[master.example_id == example_id].iloc[0]
    res = analyze_sql(ex.sql_gold, ex.dialect, SchemaIndex.from_schema(catalog[ex.schema_id]))
    d = res.get("details", {})
    print("=" * 110); print(example_id, "|", ex.source, "|", ex.db_id, "|", ex.dialect)
    print("Pregunta:", ex.question); print("SQL     :", " ".join(ex.sql_gold.split())[:400])
    print("Estado  :", res["parse_status"], "/", res["resolution_status"], "| CTE:", d.get("ctes"), "| subconsultas:", d.get("n_subqueries"), "| set ops:", res.get("set_operations"))
    print("Tablas  :", d.get("table_roles"))
    print("Columnas:"); [print(f"   {k:60s} {v}") for k, v in d.get("column_roles", {}).items()]
    if d.get("unresolved"): print("No resuelto:", d["unresolved"])
    if d.get("literal_fallbacks"): print("Literales SQLite:", d["literal_fallbacks"])

show_analysis("SPIDER-train-00001")                      # SELECT + ORDER BY
show_analysis("SPIDER-dev-00010")                        # JOIN con alias
show_analysis("SPIDER-train-00019")                      # Subconsulta

SPIDER-train-00001 | SPIDER | department_management | sqlite
Pregunta: List the name, born state and age of the heads of departments ordered by age.
SQL     : SELECT name , born_state , age FROM head ORDER BY age
Estado  : ok / resolved | CTE: [] | subconsultas: 0 | set ops: []
Tablas  : {'head': ['FROM']}
Columnas:
   head.age                                                     ['SELECT']
   head.born_state                                              ['SELECT']
   head.name                                                    ['SELECT']
SPIDER-dev-00010 | SPIDER | concert_singer | sqlite
Pregunta: Show all countries and the number of singers in each country.
SQL     : SELECT country , count(*) FROM singer GROUP BY country
Estado  : ok / resolved | CTE: [] | subconsultas: 0 | set ops: []
Tablas  : {'singer': ['FROM']}
Columnas:
   singer.Country                                               ['GROUP_BY', 'SELECT']
SPIDER-train-00019 | SPIDER | farm | sqlite
Pregunta: What is the total ho

In [5]:
geo = master[(master.source == "SPIDER") & master.sql_gold.str.contains('"')].iloc[0].example_id
show_analysis(geo)                                       # Literales entre comillas dobles (SQLite)
show_analysis("BIRD-dev-00000")                          # Columnas con espacios (BIRD)
show_analysis("BIRD-train-01833")                        # UNION dentro de tabla derivada + alias de proyección

SPIDER-train-00054 | SPIDER | farm | sqlite
Pregunta: Show the census ranking of cities whose status are not "Village".
SQL     : SELECT Census_Ranking FROM city WHERE Status != "Village"
Estado  : ok / resolved | CTE: [] | subconsultas: 0 | set ops: []
Tablas  : {'city': ['FROM']}
Columnas:
   city.Census_Ranking                                          ['SELECT']
   city.Status                                                  ['WHERE']
Literales SQLite: ['"Village"']
BIRD-dev-00000 | BIRD | california_schools | sqlite
Pregunta: What is the highest eligible free rate for K-12 students in the schools in Alameda County?
SQL     : SELECT `Free Meal Count (K-12)` / `Enrollment (K-12)` FROM frpm WHERE `County Name` = 'Alameda' ORDER BY (CAST(`Free Meal Count (K-12)` AS REAL) / `Enrollment (K-12)`) DESC LIMIT 1
Estado  : ok / resolved | CTE: [] | subconsultas: 0 | set ops: []
Tablas  : {'frpm': ['FROM']}
Columnas:
   frpm.County Name                                             ['WHERE']
   

In [6]:
show_analysis("CATASTRO-catastro-00000")               # CTE + LEFT JOIN (PostgreSQL)
show_analysis("CATASTRO-catastro-00079")               # COUNT(*) FILTER + GROUP BY + HAVING

CATASTRO-catastro-00000 | CATASTRO | catastro_unidad_intervencion_18 | postgres
Pregunta: Predios de condición NPH que en los campos 22 a 30 del número predial nacional no terminan como 000000000
SQL     : WITH tb1 AS ( SELECT ip.t_id, ip.numero_predial, ic.ilicode AS condicionpredio, SUBSTRING(ip.numero_predial, 22, 1) AS pos22, SUBSTRING(ip.numero_predial, 23, 8) AS pos23_30, SUBSTRING(ip.numero_predial, 22, 9) AS pos22_30, SUBSTRING(ip.numero_predial, 22, 4) AS pos22_25, SUBSTRING(ip.numero_predial, 26, 4) AS pos26_29 FROM unidad_intervencion_18.lc_predio ip JOIN unidad_intervencion_18.lc_condici
Estado  : ok / resolved | CTE: ['tb1'] | subconsultas: 0 | set ops: []
Tablas  : {'lc_condicionprediotipo': ['FROM', 'JOIN'], 'lc_estructuranovedadnumeropredial': ['FROM', 'JOIN'], 'lc_estructuranovedadnumeropredial_tipo_novedad': ['FROM', 'JOIN'], 'lc_predio': ['FROM', 'JOIN']}
Columnas:
   lc_condicionprediotipo.ilicode                               ['SELECT', 'WHERE']
   lc_condicionpred

### Caso de ambigüedad no resoluble (ejemplo sintético)

In [7]:
idx = SchemaIndex.from_schema(catalog["SPIDER:concert_singer"])
r = analyze_sql("SELECT Name FROM stadium, singer", "sqlite", idx)
print("Estado:", r["resolution_status"]); print("Tablas:", r["details"]["table_roles"]); print("Columnas:", r["details"]["column_roles"]); print("No resuelto:", r["details"]["unresolved"])

Estado: partial
Tablas: {'singer': ['FROM', 'JOIN'], 'stadium': ['FROM', 'JOIN']}
Columnas: {}
No resuelto: [{'kind': 'column', 'raw': 'Name', 'reason': 'ambiguous', 'clause': 'SELECT', 'candidates': ['singer.Name', 'stadium.Name']}]


## 4. Análisis de todo el dataset → `sql_analysis.parquet`

In [8]:
analysis = analyze_dataset(master, catalog, save=True)
pd.crosstab(analysis.source, [analysis.parse_status, analysis.resolution_status])

2026-11-06 22:29:25,460 | INFO | src.sql_parser | Analizadas 2000/19076 consultas
2026-11-06 22:29:27,394 | INFO | src.sql_parser | Analizadas 4000/19076 consultas
2026-11-06 22:29:29,253 | INFO | src.sql_parser | Analizadas 6000/19076 consultas
2026-11-06 22:29:31,023 | INFO | src.sql_parser | Analizadas 8000/19076 consultas
2026-11-06 22:29:33,701 | INFO | src.sql_parser | Analizadas 10000/19076 consultas
2026-11-06 22:29:36,434 | INFO | src.sql_parser | Analizadas 12000/19076 consultas
2026-11-06 22:29:38,899 | INFO | src.sql_parser | Analizadas 14000/19076 consultas
2026-11-06 22:29:41,282 | INFO | src.sql_parser | Analizadas 16000/19076 consultas
2026-11-06 22:29:43,983 | INFO | src.sql_parser | Analizadas 18000/19076 consultas
2026-11-06 22:29:45,608 | INFO | src.sql_parser | Analizadas 19076/19076 consultas
2026-11-06 22:29:45,712 | INFO | src.sql_parser | Estado de parseo:
source    parse_status
BIRD      ok              10962
CATASTRO  ok                 80
SPIDER    ok       

parse_status           ok                    
resolution_status partial resolved unresolved
source                                       
BIRD                  315    10613         34
CATASTRO                0       80          0
SPIDER                  1     8033          0

In [9]:
from collections import Counter
reasons = Counter()
for r in analysis[analysis.resolution_status != "resolved"].itertuples():
    for u in json.loads(r.analysis_json)["unresolved"]:
        reasons[(r.source, r.db_id, u["kind"], u["reason"].split(":")[0])] += 1
unres = pd.DataFrame([{"source": k[0], "db_id": k[1], "kind": k[2], "reason": k[3], "n_refs": v} for k, v in reasons.items()]).sort_values("n_refs", ascending=False)
print("Referencias no resueltas por base y motivo:")
unres.head(15)

Referencias no resueltas por base y motivo:


,source,db_id,kind,reason,n_refs
10,BIRD,retail_world,column,alias_not_found,412
11,BIRD,retail_world,column,column_not_in_table,307
8,BIRD,retail_world,column,column_not_found,208
9,BIRD,retail_world,table,table_not_in_schema,189
13,BIRD,works_cycles,column,alias_not_found,13
12,BIRD,works_cycles,table,table_not_in_schema,10
7,BIRD,app_store,column,column_not_in_table,7
14,BIRD,works_cycles,column,column_not_found,4
6,BIRD,shooting,column,column_not_in_table,2
5,BIRD,shooting,column,column_not_found,2


Interpretación: prácticamente todas las referencias no resueltas provienen de la base BIRD `retail_world` (Northwind), cuyo
`train_tables.json` describe una versión reducida del esquema (faltan `EmployeeTerritories`, `Territories` y columnas como `CompanyName`,
`UnitPrice`...), y de `shooting`/`app_store`/`works_cycles` por columnas ausentes en la metadata. En Spider solo falla `SPIDER-train-03153`
(`Ref_Company_Types` no existe en `assets_maintenance`), error conocido del benchmark. No se inventan elementos: estas consultas quedan marcadas para revisión manual.


In [10]:
print("Métodos de resolución de columnas (trazabilidad del parser):")
meth = Counter()
for r in analysis.itertuples():
    for c in json.loads(r.analysis_json).get("column_refs", []):
        meth[(r.source, c["method"])] += 1
pd.Series(meth).rename("n").reset_index().rename(columns={"level_0": "source", "level_1": "method"}).pivot(index="method", columns="source", values="n").fillna(0).astype(int)

Métodos de resolución de columnas (trazabilidad del parser):


source,BIRD,CATASTRO,SPIDER
method,,,
literal_fallback,2,0,2215
projection_alias,3,0,0
qualified_derived,347,54,0
qualified_derived_star,6,0,0
qualified_physical,48645,934,16710
qualified_physical_outer,5,0,2
unqualified_derived,185,23,0
unqualified_unique,7866,21,11613
unqualified_unique_outer,3,0,0


## 5. Generación de etiquetas de relevancia y roles

Unidad de anotación = (pregunta, elemento del esquema). Para cada pregunta se generan **todas** las tablas y columnas de su esquema
(positivos y negativos). `relevance = 1` si el parser identificó al elemento en el SQL; `roles` = vector binario de 7 posiciones.

Estados: `automatic` (consulta completamente resuelta), `manual_review` (consulta con referencias no resueltas: los positivos son
correctos pero los negativos podrían ser falsos), `unresolved` (candidato de una referencia ambigua o SQL no parseable), `validated` (reservado para revisión humana).

In [9]:
from src.schema_annotator import build_annotations, example_annotation_view

annotations = build_annotations(master, analysis, catalog, save=True)
pd.DataFrame({s: {"anotaciones": len(a), "preguntas": a.example_id.nunique(), "relevantes": int(a.relevance.sum()),
                  "tasa_relevancia_%": round(a.relevance.mean() * 100, 2), **a.annotation_status.value_counts().to_dict()}
              for s, a in annotations.items()}).T

2026-11-06 22:36:40,545 | INFO | src.schema_annotator | SPIDER: 299736 anotaciones (8034 preguntas, 0 sin análisis) | relevancia=1: 35662 | estados: {'automatic': 299658, 'manual_review': 78}
2026-11-06 22:36:41,002 | INFO | src.schema_annotator | Guardado data/annotations/spider_annotations.parquet
2026-11-06 22:36:51,127 | INFO | src.schema_annotator | BIRD: 963578 anotaciones (10962 preguntas, 0 sin análisis) | relevancia=1: 71297 | estados: {'automatic': 941694, 'manual_review': 21884}
2026-11-06 22:36:52,557 | INFO | src.schema_annotator | Guardado data/annotations/bird_annotations.parquet
2026-11-06 22:36:53,884 | INFO | src.schema_annotator | CATASTRO: 164240 anotaciones (80 preguntas, 0 sin análisis) | relevancia=1: 911 | estados: {'automatic': 164240}
2026-11-06 22:36:54,154 | INFO | src.schema_annotator | Guardado data/annotations/catastro_annotations.parquet


,anotaciones,preguntas,relevantes,tasa_relevancia_%,automatic,manual_review
SPIDER,299736.0,8034.0,35662.0,11.90,299658.0,78.0
BIRD,963578.0,10962.0,71297.0,7.40,941694.0,21884.0
CATASTRO,164240.0,80.0,911.0,0.55,164240.0,NaN


In [10]:
print(master.set_index("example_id").loc["SPIDER-dev-00010", ["question", "sql_gold"]].to_dict())
example_annotation_view(annotations["SPIDER"], "SPIDER-dev-00010")

{'question': 'Show all countries and the number of singers in each country.', 'sql_gold': 'SELECT country ,  count(*) FROM singer GROUP BY country'}


,element_id,element_type,relevance,role_SELECT,role_WHERE,role_JOIN,role_FROM,role_GROUP_BY,role_HAVING,role_ORDER_BY,annotation_status,annotation_notes
0,singer,table,1,0,0,0,1,0,0,0,automatic,star_projection
1,singer.Country,column,1,1,0,0,0,1,0,0,automatic,NaN


In [11]:
print(master.set_index("example_id").loc["BIRD-dev-00000", ["question", "sql_gold"]].to_dict())
example_annotation_view(annotations["BIRD"], "BIRD-dev-00000")

{'question': 'What is the highest eligible free rate for K-12 students in the schools in Alameda County?', 'sql_gold': "SELECT `Free Meal Count (K-12)` / `Enrollment (K-12)` FROM frpm WHERE `County Name` = 'Alameda' ORDER BY (CAST(`Free Meal Count (K-12)` AS REAL) / `Enrollment (K-12)`) DESC LIMIT 1"}


,element_id,element_type,relevance,role_SELECT,role_WHERE,role_JOIN,role_FROM,role_GROUP_BY,role_HAVING,role_ORDER_BY,annotation_status,annotation_notes
0,frpm,table,1,0,0,0,1,0,0,0,automatic,NaN
1,frpm.County Name,column,1,0,1,0,0,0,0,0,automatic,NaN
2,frpm.Enrollment (K-12),column,1,1,0,0,0,0,0,1,automatic,NaN
3,frpm.Free Meal Count (K-12),column,1,1,0,0,0,0,0,1,automatic,NaN


In [12]:
print(master.set_index("example_id").loc["CATASTRO-catastro-00079", ["question"]].to_dict())
example_annotation_view(annotations["CATASTRO"], "CATASTRO-catastro-00079")

{'question': 'Predios que registran simultáneamente novedades de número predial de tipo Englobe y Desenglobe, o simultáneamente novedades de tipo Cancelacion y Predio_Nuevo'}


,element_id,element_type,relevance,role_SELECT,role_WHERE,role_JOIN,role_FROM,role_GROUP_BY,role_HAVING,role_ORDER_BY,annotation_status,annotation_notes
0,lc_datosadicionaleslevantamientocatastral,table,1,0,0,1,1,0,0,0,automatic,star_projection
1,lc_estructuranovedadnumeropredial,table,1,0,0,1,1,0,0,0,automatic,star_projection
2,lc_estructuranovedadnumeropredial_tipo_novedad,table,1,0,0,1,1,0,0,0,automatic,star_projection
3,lc_predio,table,1,0,0,1,1,0,0,0,automatic,star_projection
4,lc_datosadicionaleslevantamientocatastral.t_id,column,1,0,0,1,0,0,0,0,automatic,NaN
5,lc_datosadicionaleslevantamientocatastral.lc_predio,column,1,0,0,1,0,0,0,0,automatic,NaN
6,lc_estructuranovedadnumeropredial.tipo_novedad,column,1,0,0,1,0,0,0,0,automatic,NaN
7,lc_estructuranovedadnumeropredial.lc_dtsdcnlstmntctstral_novedad_numeros_prediales,column,1,0,0,1,0,0,0,0,automatic,NaN
8,lc_estructuranovedadnumeropredial_tipo_novedad.t_id,column,1,0,0,1,0,0,0,0,automatic,NaN
9,lc_estructuranovedadnumeropredial_tipo_novedad.ilicode,column,1,1,0,0,0,0,1,0,automatic,NaN


In [15]:
print("Candidatos (positivos + negativos) de una pregunta Spider, incluyendo los no relevantes:")
example_annotation_view(annotations["SPIDER"], "SPIDER-train-00000", only_relevant=False)

Candidatos (positivos + negativos) de una pregunta Spider, incluyendo los no relevantes:


,element_id,element_type,relevance,role_SELECT,role_WHERE,role_JOIN,role_FROM,role_GROUP_BY,role_HAVING,role_ORDER_BY,annotation_status,annotation_notes
0,department,table,0,0,0,0,0,0,0,0,automatic,NaN
1,head,table,1,0,0,0,1,0,0,0,automatic,star_projection
2,management,table,0,0,0,0,0,0,0,0,automatic,NaN
3,department.Department_ID,column,0,0,0,0,0,0,0,0,automatic,NaN
4,department.Name,column,0,0,0,0,0,0,0,0,automatic,NaN
5,department.Creation,column,0,0,0,0,0,0,0,0,automatic,NaN
6,department.Ranking,column,0,0,0,0,0,0,0,0,automatic,NaN
7,department.Budget_in_Billions,column,0,0,0,0,0,0,0,0,automatic,NaN
8,department.Num_Employees,column,0,0,0,0,0,0,0,0,automatic,NaN
9,head.head_ID,column,0,0,0,0,0,0,0,0,automatic,NaN


In [13]:
print("Ejemplo marcado para revisión manual (BIRD retail_world):")
mr = annotations["BIRD"][annotations["BIRD"].annotation_status == "manual_review"].iloc[0]
print(mr.question); print(mr.sql_gold); print("Nota:", mr.annotation_notes)
example_annotation_view(annotations["BIRD"], mr.example_id)

Ejemplo marcado para revisión manual (BIRD retail_world):
Please list the texts of all the tweets posted by male users from Buenos Aires.
SELECT T1.text FROM twitter AS T1 INNER JOIN location AS T2 ON T2.LocationID = T1.LocationID INNER JOIN user AS T2 ON T2.UserID = T1.UserID INNER JOIN user AS T3 ON T1.UserID = T3.UserID WHERE T2.City = 'Buenos Aires' AND T3.Gender = 'Male'
Nota: unresolved_references=column:T2.UserID:column_not_in_table:location; scope:SELECT T1.text FROM twitter AS T1 INNER JOIN location AS T2 ON T2.LocationID = T:duplicate_alias: Alias already used: T2


,element_id,element_type,relevance,role_SELECT,role_WHERE,role_JOIN,role_FROM,role_GROUP_BY,role_HAVING,role_ORDER_BY,annotation_status,annotation_notes
0,location,table,1,0,0,1,1,0,0,0,manual_review,unresolved_references=column:T2.UserID:column_not_in_table:location; scope:SELECT T1.text FROM twitter AS T1 INNER J...
1,user,table,1,0,0,1,1,0,0,0,manual_review,unresolved_references=column:T2.UserID:column_not_in_table:location; scope:SELECT T1.text FROM twitter AS T1 INNER J...
2,twitter,table,1,0,0,1,1,0,0,0,manual_review,unresolved_references=column:T2.UserID:column_not_in_table:location; scope:SELECT T1.text FROM twitter AS T1 INNER J...
3,location.LocationID,column,1,0,0,1,0,0,0,0,manual_review,unresolved_references=column:T2.UserID:column_not_in_table:location; scope:SELECT T1.text FROM twitter AS T1 INNER J...
4,location.City,column,1,0,1,0,0,0,0,0,manual_review,unresolved_references=column:T2.UserID:column_not_in_table:location; scope:SELECT T1.text FROM twitter AS T1 INNER J...
5,user.UserID,column,1,0,0,1,0,0,0,0,manual_review,unresolved_references=column:T2.UserID:column_not_in_table:location; scope:SELECT T1.text FROM twitter AS T1 INNER J...
6,user.Gender,column,1,0,1,0,0,0,0,0,manual_review,unresolved_references=column:T2.UserID:column_not_in_table:location; scope:SELECT T1.text FROM twitter AS T1 INNER J...
7,twitter.text,column,1,1,0,0,0,0,0,0,manual_review,unresolved_references=column:T2.UserID:column_not_in_table:location; scope:SELECT T1.text FROM twitter AS T1 INNER J...
8,twitter.LocationID,column,1,0,0,1,0,0,0,0,manual_review,unresolved_references=column:T2.UserID:column_not_in_table:location; scope:SELECT T1.text FROM twitter AS T1 INNER J...
9,twitter.UserID,column,1,0,0,1,0,0,0,0,manual_review,unresolved_references=column:T2.UserID:column_not_in_table:location; scope:SELECT T1.text FROM twitter AS T1 INNER J...


## 6. Resumen

* 100 % de las 19.076 consultas parseadas con SQLGlot (0 errores de sintaxis).
* Resolución completa: Spider 8.033/8.034, BIRD 10.613/10.962, Catastro 80/80.
* Anotaciones generadas: `spider_annotations.parquet` (299.736), `bird_annotations.parquet` (963.578), `catastro_annotations.parquet` (164.240).
* Los casos no resueltos quedan contabilizados en `sql_analysis.parquet` y marcados como `manual_review` en las anotaciones (notebook 04).


In [21]:
df = pd.read_parquet(r"D:\backup\MAESTRIA\MINE\2026_2\ProcesamientoLenguajeNatural\Tareas\proyecto_jarm\data\annotations\bird_annotations.parquet")
df

,example_id,source,db_id,split,question,sql_gold,element_id,element_name,element_type,table_name,relevance,roles,role_SELECT,role_WHERE,role_JOIN,role_FROM,role_GROUP_BY,role_HAVING,role_ORDER_BY,annotation_status,parser_status,annotation_notes
0,BIRD-train-00000,BIRD,movie_platform,test,Name movie titles released in year 1945. Sort the listing by the descending order of movie popularity.,SELECT movie_title FROM movies WHERE movie_release_year = 1945 ORDER BY movie_popularity DESC LIMIT 1,lists,lists,table,lists,0,"[0, 0, 0, 0, 0, 0, 0]",0,0,0,0,0,0,0,automatic,resolved,NaN
1,BIRD-train-00000,BIRD,movie_platform,test,Name movie titles released in year 1945. Sort the listing by the descending order of movie popularity.,SELECT movie_title FROM movies WHERE movie_release_year = 1945 ORDER BY movie_popularity DESC LIMIT 1,movies,movies,table,movies,1,"[0, 0, 0, 1, 0, 0, 0]",0,0,0,1,0,0,0,automatic,resolved,NaN
2,BIRD-train-00000,BIRD,movie_platform,test,Name movie titles released in year 1945. Sort the listing by the descending order of movie popularity.,SELECT movie_title FROM movies WHERE movie_release_year = 1945 ORDER BY movie_popularity DESC LIMIT 1,ratings_users,ratings_users,table,ratings_users,0,"[0, 0, 0, 0, 0, 0, 0]",0,0,0,0,0,0,0,automatic,resolved,NaN
3,BIRD-train-00000,BIRD,movie_platform,test,Name movie titles released in year 1945. Sort the listing by the descending order of movie popularity.,SELECT movie_title FROM movies WHERE movie_release_year = 1945 ORDER BY movie_popularity DESC LIMIT 1,lists_users,lists_users,table,lists_users,0,"[0, 0, 0, 0, 0, 0, 0]",0,0,0,0,0,0,0,automatic,resolved,NaN
4,BIRD-train-00000,BIRD,movie_platform,test,Name movie titles released in year 1945. Sort the listing by the descending order of movie popularity.,SELECT movie_title FROM movies WHERE movie_release_year = 1945 ORDER BY movie_popularity DESC LIMIT 1,ratings,ratings,table,ratings,0,"[0, 0, 0, 0, 0, 0, 0]",0,0,0,0,0,0,0,automatic,resolved,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
963573,BIRD-dev-01533,BIRD,debit_card_specializing,validation,For all the people who paid more than 29.00 per unit of product id No.5. Give their consumption status in the August...,SELECT T2.Consumption FROM transactions_1k AS T1 INNER JOIN yearmonth AS T2 ON T1.CustomerID = T2.CustomerID WHERE T...,transactions_1k.Amount,Amount,column,transactions_1k,1,"[0, 1, 0, 0, 0, 0, 0]",0,1,0,0,0,0,0,automatic,resolved,NaN
963574,BIRD-dev-01533,BIRD,debit_card_specializing,validation,For all the people who paid more than 29.00 per unit of product id No.5. Give their consumption status in the August...,SELECT T2.Consumption FROM transactions_1k AS T1 INNER JOIN yearmonth AS T2 ON T1.CustomerID = T2.CustomerID WHERE T...,transactions_1k.Price,Price,column,transactions_1k,1,"[0, 1, 0, 0, 0, 0, 0]",0,1,0,0,0,0,0,automatic,resolved,NaN
963575,BIRD-dev-01533,BIRD,debit_card_specializing,validation,For all the people who paid more than 29.00 per unit of product id No.5. Give their consumption status in the August...,SELECT T2.Consumption FROM transactions_1k AS T1 INNER JOIN yearmonth AS T2 ON T1.CustomerID = T2.CustomerID WHERE T...,yearmonth.CustomerID,CustomerID,column,yearmonth,1,"[0, 0, 1, 0, 0, 0, 0]",0,0,1,0,0,0,0,automatic,resolved,NaN
963576,BIRD-dev-01533,BIRD,debit_card_specializing,validation,For all the people who paid more than 29.00 per unit of product id No.5. Give their consumption status in the August...,SELECT T2.Consumption FROM transactions_1k AS T1 INNER JOIN yearmonth AS T2 ON T1.CustomerID = T2.CustomerID WHERE T...,yearmonth.Date,Date,column,yearmonth,1,"[0, 1, 0, 0, 0, 0, 0]",0,1,0,0,0,0,0,automatic,resolved,NaN
